# Glunova — DFU Detection & Segmentation

> **Project:** Glunova AI Platform — Clinic Decision Support module  
> **Task:** Binary semantic segmentation — detect & delineate ulcer regions from clinical photographs  
> **XAI:** Segmentation-native explainability (Score-CAM, Attention Rollout, pixel-attribution RISE)

---

## What is this notebook?

**Diabetic Foot Ulcers (DFU)** are open wounds on the feet of people with diabetes.  
This notebook trains and benchmarks multiple AI models to **automatically segment ulcer regions** from clinical photographs, then explains their decisions with clinician-oriented visualisations.

---

## Model Roster

| # | Model | Backbone | Pre-trained on | Params | Role |
|---|-------|----------|---------------|--------|------|
| 1 | **ResNet-34 U-Net** | ResNet-34 (CNN) | ImageNet-1k | ~24 M | Fast CNN baseline |
| 2 | **SegFormer-B2** | MiT-B2 (hierarchical ViT) | ImageNet-1k | ~27 M | Efficient transformer; FUS-Challenge SOTA-tier |

### Why these two?

| Model | Strengths for DFU | Caveats |
|-------|-------------------|---------|
| ResNet-34 U-Net | Fast, well-understood skip-connections, GPU-efficient | Pure CNN — limited long-range context |
| SegFormer-B2 | Hierarchical ViT captures both texture & global anatomy; lightweight MLP decoder | Supervised ImageNet pre-training only |

### Why NOT others?

| Alternative | Why excluded |
|-------------|-------------|
| Swin-Base (~88 M) | 3× compute vs SegFormer for ~1 % Dice gain at this data scale |
| SAM / MedSAM | Prompt-based, not end-to-end trainable for pixel segmentation without custom heads |
| nnU-Net | Designed for volumetric / multi-modal data; overkill for single-image 2-D DFU |
| SegFormer-B3/B4 | 40–60 % more compute for <1 % Dice gain here |

---

## End-to-End Pipeline

```
Raw clinical images + masks
(Foot Ulcer Segmentation Challenge)
   →
[0] Env Setup (GPU, deps, paths)
   →
[1] Data Loading (train / val / test)
   →
[2] EDA (stats, class balance)
   →
[3] Utilities (Dataset, Albumentations, Loss, TTA, loop)
   →
[4] Models (U-Net R34 | SegFormer-B2)
   →
[5] Training (flags, optional skip)
   →
[6] Eval + TTA + Thresholding (best model)
   →
[7] Inference + Failure Analysis
   →
[8] XAI (Score-CAM, Rollout, RISE, reports)
   →
[9] Export (glunova_dfu_export.zip)
```


---
## §0 · Environment Setup

In [ ]:
# ── 0.1  GPU check ────────────────────────────────────────────────────────────
import os, sys, warnings

os.environ.setdefault("PYTHONWARNINGS", "ignore")
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")

import torch
torch.set_warn_always(False)

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU ok  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  ↳ {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("  ⚠️  No GPU detected — attach a T4/P100/A100 runtime before training.")


In [ ]:
# ── 0.2  Install dependencies ─────────────────────────────────────────────────
import os, subprocess, sys, warnings

os.environ.setdefault("PYTHONWARNINGS", "ignore")
warnings.simplefilter("ignore")

packages = [
    "pycocotools",
    "opencv-python-headless",
    "matplotlib", "seaborn", "tqdm",
    "scikit-learn",
    "onnx", "onnxruntime-gpu",
    "transformers>=4.40.0",
    "timm>=0.9.12",
    "grad-cam>=1.4.8",
    "reportlab>=4.1.0",
    "segmentation-models-pytorch>=0.3.3",
    "accelerate",
    "albumentations>=1.3.1",
    "scipy",
]

ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    capture_output=True, text=True,
)
if ret.returncode != 0:
    print("⚠️  Some packages may have failed:")
    print(ret.stderr[-2000:])
else:
    print("✓ All packages installed successfully.")


In [ ]:
# ── 0.3  Global imports & reproducibility ─────────────────────────────────────
import os

os.environ["PYTHONWARNINGS"]         = "ignore"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, shutil, random, warnings, logging, base64, datetime, io
from pathlib import Path
from copy import deepcopy

warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)
for _log in ("matplotlib", "matplotlib.font_manager", "PIL", "urllib3",
             "h5py", "albumentations", "timm"):
    logging.getLogger(_log).setLevel(logging.ERROR)

import numpy as np
np.seterr(all="ignore")
import cv2
import torch
torch.set_warn_always(False)
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerConfig,
)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # Faster convs on fixed input sizes; slightly less strict reproducibility than deterministic=True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")


In [ ]:
# ── 0.4  Project directory structure ──────────────────────────────────────────
ROOT       = Path("/kaggle/working/glunova_dfu")
OUTPUT_DIR = ROOT / "output"
VIZ_DIR    = ROOT / "viz"
XAI_DIR    = ROOT / "xai"
CKPT_DIR   = ROOT / "checkpoints"

for d in [OUTPUT_DIR, VIZ_DIR, XAI_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

KAGGLE_INPUT = Path("/kaggle/input/datasets/yessinehakim1/dfu-seg-preped/dfu seg preped")

if not KAGGLE_INPUT.exists():
    raise FileNotFoundError("Dataset not found. Check KAGGLE_INPUT path.")

print(f"Dataset root  : {KAGGLE_INPUT}")
print(f"Working root  : {ROOT}")
print("Directories  :", [d.name for d in ROOT.iterdir()])


---
## §1 · Dataset Loading — Foot Ulcer Segmentation Challenge

| Property | Value |
|----------|-------|
| Splits | Pre-defined **train / val / test** |
| Folder layout | `{split}/images/`, `{split}/masks/` |
| Mask format | Binary PNG — `0 = background`, `255 = ulcer` |
| Re-splitting | **Not done** — preserves FUS Challenge leaderboard comparability |


In [ ]:
# ── 1.1  Discover dataset structure ───────────────────────────────────────────
expected = {
    "train": ["images", "masks"],
    "val":   ["images", "masks"],
    "test":  ["images", "masks"],
}

print(f"Dataset root: {KAGGLE_INPUT}")
for split, subdirs in expected.items():
    base = KAGGLE_INPUT / split
    ok   = base.exists()
    print(f"  {split:5s}: {'✓ found' if ok else '✗ MISSING'}")
    if ok:
        for sd in subdirs:
            p = base / sd
            n = len(list(p.iterdir())) if p.exists() else 0
            print(f"      {sd:6s}: {'✓' if p.exists() else '✗'}  ({n} files)")


In [ ]:
# ── 1.2  Load pre-split images and masks ──────────────────────────────────────
IMG_EXTS = {".png", ".jpg", ".jpeg"}


def load_split(root: Path, split: str):
    img_dir = mask_dir = None
    for c in [root / split / "images", root / split.capitalize() / "images",
               root / split / "Images"]:
        if c.exists(): img_dir = c; break
    for c in [root / split / "masks", root / split.capitalize() / "masks",
               root / split / "Masks"]:
        if c.exists(): mask_dir = c; break

    if img_dir is None or mask_dir is None:
        raise FileNotFoundError(f"Cannot locate images/masks for '{split}' under {root}")

    img_dict  = {p.stem: p for p in img_dir.iterdir()  if p.suffix.lower() in IMG_EXTS}
    mask_dict = {p.stem: p for p in mask_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
    common    = sorted(img_dict.keys() & mask_dict.keys())

    only_imgs  = img_dict.keys() - mask_dict.keys()
    only_masks = mask_dict.keys() - img_dict.keys()
    if only_imgs:  print(f"  ⚠  {split}: {len(only_imgs)} image(s) without mask — dropped")
    if only_masks: print(f"  ⚠  {split}: {len(only_masks)} mask(s) without image — dropped")
    if not common: raise ValueError(f"No image/mask pairs for split '{split}'.")

    return [img_dict[s] for s in common], [mask_dict[s] for s in common]


print("Loading dataset splits...")
train_imgs, train_masks = load_split(KAGGLE_INPUT, "train")
val_imgs,   val_masks   = load_split(KAGGLE_INPUT, "val")
test_imgs,  test_masks  = load_split(KAGGLE_INPUT, "test")

def count_positive(masks):
    return sum(1 for m in masks if np.array(Image.open(m).convert("L")).max() > 0)

print(f"\nTrain : {len(train_imgs):4d} images  ulcer-positive: {count_positive(train_masks)}")
print(f"Val   : {len(val_imgs):4d} images  ulcer-positive: {count_positive(val_masks)}")
print(f"Test  : {len(test_imgs):4d} images  ulcer-positive: {count_positive(test_masks)}")
print("\n✓ All splits loaded.")


---
## §2 · Exploratory Data Analysis

In [ ]:
# ── 2.1  Sample images + overlays ─────────────────────────────────────────────
def overlay_mask(img_path, mask_path, alpha: float = 0.45) -> np.ndarray:
    img_pil  = Image.open(img_path).convert("RGB")
    mask_pil = Image.open(mask_path).convert("L")
    if mask_pil.size != img_pil.size:
        mask_pil = mask_pil.resize(img_pil.size, resample=Image.NEAREST)
    img  = np.array(img_pil, dtype=np.float32)
    mask = np.array(mask_pil) > 0
    overlay = img.copy()
    overlay[mask] = overlay[mask] * (1-alpha) + np.array([220, 50, 50]) * alpha
    return overlay.astype(np.uint8)

pos_pairs = [
    (i, m) for i, m in zip(train_imgs, train_masks)
    if np.array(Image.open(m)).max() > 0
][:6]

n   = len(pos_pairs)
fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
fig.suptitle("FUS Challenge — samples (top) and ulcer overlays (bottom)", fontsize=12)
if n == 1: axes = np.array(axes).reshape(2,1)

for col, (img_p, mask_p) in enumerate(pos_pairs):
    axes[0, col].imshow(Image.open(img_p));           axes[0, col].axis("off")
    axes[1, col].imshow(overlay_mask(img_p, mask_p)); axes[1, col].axis("off")

plt.tight_layout()
plt.savefig(VIZ_DIR / "sample_images.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── 2.2  Dataset statistics ────────────────────────────────────────────────────
all_images = train_imgs + val_imgs + test_imgs
all_masks  = train_masks + val_masks + test_masks

stats = {"widths": [], "heights": [], "ulcer_ratio": [], "has_ulcer": []}

for img_p, mask_p in tqdm(zip(all_images, all_masks),
                          total=len(all_images), desc="Computing stats"):
    img  = Image.open(img_p)
    mask = np.array(Image.open(mask_p))
    w, h = img.size
    stats["widths"].append(w);  stats["heights"].append(h)
    ratio = float((mask > 0).sum()) / float(mask.size)
    stats["ulcer_ratio"].append(ratio)
    stats["has_ulcer"].append(int(ratio > 0))

n_total = len(all_images)
n_ulcer = sum(stats["has_ulcer"])
print("=" * 55)
print(f"Total images      : {n_total}")
print(f"  Train/Val/Test  : {len(train_imgs)}/{len(val_imgs)}/{len(test_imgs)}")
print(f"With ulcer        : {n_ulcer} ({100*n_ulcer/n_total:.1f}%)")
print(f"Width  — mean {np.mean(stats['widths']):.0f} | min {min(stats['widths'])} | max {max(stats['widths'])}")
print(f"Height — mean {np.mean(stats['heights']):.0f} | min {min(stats['heights'])} | max {max(stats['heights'])}")
print(f"Ulcer area — mean {100*np.mean(stats['ulcer_ratio']):.2f}% | median {100*np.median(stats['ulcer_ratio']):.2f}%")
print("=" * 55)


In [ ]:
# ── 2.3  Distribution plots ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(stats["widths"], stats["heights"], alpha=0.3, s=10, color="steelblue")
axes[0].set_xlabel("Width (px)"); axes[0].set_ylabel("Height (px)")
axes[0].set_title("Image dimensions")

pos_ratios = [r for r, h in zip(stats["ulcer_ratio"], stats["has_ulcer"]) if h]
axes[1].hist([r*100 for r in pos_ratios], bins=40, color="#C0570D", edgecolor="white")
axes[1].set_xlabel("Ulcer area (% of image)"); axes[1].set_ylabel("Count")
axes[1].set_title("Ulcer area distribution (ulcer-positive only)")

split_labels = ["Train", "Val", "Test"]
split_counts = [len(train_imgs), len(val_imgs), len(test_imgs)]
axes[2].bar(split_labels, split_counts,
            color=["#2E6DA4","#C0570D","#1B8A3F"], edgecolor="white")
for i, c in enumerate(split_counts):
    axes[2].text(i, c+3, str(c), ha="center", fontsize=11)
axes[2].set_title("Split sizes")

plt.suptitle("Foot Ulcer Segmentation Challenge — EDA", fontsize=13)
plt.tight_layout()
plt.savefig(VIZ_DIR / "eda_plots.png", dpi=120, bbox_inches="tight")
plt.show()


---
## §3 · Shared utilities

All models share:

- **`FUSDataset`** — Albumentations-backed dataset with rich medical-imaging augmentations  
- **Training loss** — `dice_bce_loss`: 50% Dice + 50% BCEWithLogits (optional `focal_w` > 0 to add Focal); **`combined_loss`** = 60% Dice + 40% Focal also available  
- **`predict_tta`** — 3-way Test-Time Augmentation (original + h-flip + v-flip)  
- **`train_model`** — AMP, cosine LR, gradient accumulation, early stopping (patience=10); optional EMA, aux loss, grad clipping

### Why Dice + BCE (+ optional Focal)?

| Loss | Behaviour | DFU fit |
|------|-----------|---------|
| BCE | Pixel-wise calibration with logits | Stable gradients on dense regions |
| **Dice** | Maximises overlap coefficient | Aligns with the segmentation metric |
| **Focal** (optional) | Down-weights easy negatives via (1−p)^γ | Helps heavy mask imbalance / hard boundaries when needed |


In [ ]:
# ── 3.1  ImageNet normalisation stats ─────────────────────────────────────────
IMGNET_MEAN = [0.485, 0.456, 0.406]
IMGNET_STD  = [0.229, 0.224, 0.225]


# ── 3.2  Albumentations pipelines ─────────────────────────────────────────────

def get_train_transform(img_size: int):
    """
    Rich augmentation pipeline for clinical DFU images.

    Justification:
      ElasticTransform  — simulates soft-tissue deformation (standard in med-seg)
      GridDistortion    — adds local geometric variance
      CLAHE             — improves local contrast for wound texture
      GaussNoise        — mimics sensor/lighting noise in clinical photos
      CoarseDropout     — forces partial-view reasoning (occlusion robustness)
      ColorJitter       — via RandomBrightnessContrast + HueSaturationValue
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.12, rotate_limit=18,
            border_mode=cv2.BORDER_REFLECT_101, p=0.55,
        ),
        A.ElasticTransform(alpha=80, sigma=80*0.05,
                           alpha_affine=80*0.03, p=0.35),
        A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3),
        A.OpticalDistortion(distort_limit=0.08, shift_limit=0.05, p=0.2),
        A.RandomBrightnessContrast(brightness_limit=0.3,
                                   contrast_limit=0.3, p=0.7),
        A.HueSaturationValue(hue_shift_limit=8,
                             sat_shift_limit=20, val_shift_limit=20, p=0.5),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.GaussianBlur(blur_limit=3, p=0.2),
        A.CoarseDropout(max_holes=6, max_height=32, max_width=32,
                        fill_value=0, mask_fill_value=0, p=0.2),
        A.Normalize(mean=IMGNET_MEAN, std=IMGNET_STD),
        ToTensorV2(),
    ])


def get_val_transform(img_size: int):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMGNET_MEAN, std=IMGNET_STD),
        ToTensorV2(),
    ])


In [ ]:
# ── 3.3  FUSDataset ───────────────────────────────────────────────────────────

class FUSDataset(Dataset):
    """
    Foot Ulcer Segmentation Challenge dataset.
    Returns (image_tensor [3×H×W], mask_tensor [1×H×W]).
    """

    def __init__(self, image_paths, mask_paths,
                 img_size: int = 512, augment: bool = False):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = (get_train_transform(img_size)
                            if augment else get_val_transform(img_size))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img  = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        mask = np.array(Image.open(self.mask_paths[idx]).convert("L"))

        augmented = self.transform(image=img, mask=mask)
        img_t  = augmented["image"].float()
        mask_t = (augmented["mask"] > 127).float()
        return img_t, mask_t.unsqueeze(0)


def make_loaders(img_size: int, batch_size: int):
    train_ds = FUSDataset(train_imgs, train_masks, img_size, augment=True)
    val_ds   = FUSDataset(val_imgs,   val_masks,   img_size, augment=False)
    test_ds  = FUSDataset(test_imgs,  test_masks,  img_size, augment=False)

    nw = min(4, max(1, (os.cpu_count() or 2) - 1))
    pin = torch.cuda.is_available()
    kw = dict(num_workers=nw, pin_memory=pin, persistent_workers=(nw > 0))
    if nw > 0:
        kw["prefetch_factor"] = 2
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  **kw)
    vl = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, **kw)
    sl = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, **kw)
    return tl, vl, sl


print("FUSDataset with Albumentations defined.")


In [ ]:
# ── 3.4  Loss functions ────────────────────────────────────────────────────────

def dice_loss(pred: torch.Tensor, target: torch.Tensor,
              smooth: float = 1.0) -> torch.Tensor:
    pred   = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    inter  = (pred * target).sum()
    return 1.0 - (2.0 * inter + smooth) / (pred.sum() + target.sum() + smooth)


def focal_loss(logits: torch.Tensor, target: torch.Tensor,
               alpha: float = 0.8, gamma: float = 2.0) -> torch.Tensor:
    """
    Binary Focal loss.
    alpha : weight for positive class (ulcer); mitigates class imbalance
    gamma : focusing exponent — higher → steeper down-weighting of easy pixels
    """
    probs   = torch.sigmoid(logits)
    bce_raw = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    pt      = torch.where(target == 1, probs, 1.0 - probs)
    focal_w = alpha * (1.0 - pt) ** gamma
    return (focal_w * bce_raw).mean()


def combined_loss(logits: torch.Tensor, target: torch.Tensor,
                  dice_w: float = 0.6, focal_w: float = 0.4) -> torch.Tensor:
    """60 % Dice + 40 % Focal (alternative recipe)."""
    probs = torch.sigmoid(logits)
    dl    = dice_loss(probs, target)
    fl    = focal_loss(logits, target)
    return dice_w * dl + focal_w * fl


def dice_bce_loss(logits: torch.Tensor, target: torch.Tensor,
                  dice_w: float = 0.5, bce_w: float = 0.5,
                  focal_w: float = 0.0) -> torch.Tensor:
    """
    Primary objective: Dice + BCEWithLogits.
    Set focal_w > 0 to add a Focal term when foreground/background imbalance is extreme.
    """
    probs = torch.sigmoid(logits)
    dl = dice_loss(probs, target)
    bce = F.binary_cross_entropy_with_logits(logits, target)
    loss = dice_w * dl + bce_w * bce
    if focal_w > 0:
        loss = loss + focal_w * focal_loss(logits, target)
    return loss


print("Loss functions defined (Dice + BCEWithLogits; Focal available; combined_loss = Dice + Focal).")


In [ ]:
from typing import Optional


def compute_binary_seg_metrics(pred: np.ndarray, gt: np.ndarray) -> tuple:
    """
    Per-image binary metrics (ulcer = positive class).
    Returns: dice, iou, sensitivity (recall), specificity, precision, f1.
    Undefined rates (e.g. sensitivity when GT has no foreground) are np.nan.
    """
    pred = (pred > 0).astype(np.uint8)
    gt   = (gt   > 0).astype(np.uint8)

    inter = int((pred & gt).sum())
    union = int((pred | gt).sum())
    p_sum = int(pred.sum())
    g_sum = int(gt.sum())

    tp = inter
    fp = int((pred & (1 - gt)).sum())
    fn = int(((1 - pred) & gt).sum())
    tn = int(((1 - pred) & (1 - gt)).sum())

    if p_sum == 0 and g_sum == 0:
        dice, iou = 1.0, 1.0
    else:
        dice = 2 * inter / (p_sum + g_sum + 1e-6)
        iou  = inter / (union + 1e-6)

    sens = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    prec = tp / (tp + fp) if (tp + fp) > 0 else float("nan")

    if p_sum == 0 and g_sum == 0:
        f1 = 1.0
    elif (2 * tp + fp + fn) == 0:
        f1 = float("nan")
    else:
        f1 = (2 * tp) / (2 * tp + fp + fn)

    return float(dice), float(iou), float(sens), float(spec), float(prec), float(f1)


def compute_dice_iou(pred: np.ndarray, gt: np.ndarray) -> tuple:
    d, i, _, _, _, _ = compute_binary_seg_metrics(pred, gt)
    return d, i

@torch.no_grad()
def predict_tta(model: nn.Module,
                img_t: torch.Tensor,
                device: str) -> torch.Tensor:
    """
    3-way TTA:
    - original
    - horizontal flip
    - vertical flip
    → averaged probability map
    """
    model.eval()
    pin = torch.cuda.is_available()
    img_t = img_t.to(device, non_blocking=pin)

    p0 = torch.sigmoid(model(img_t))

    p1 = torch.sigmoid(
        model(torch.flip(img_t, dims=[3]))
    ).flip(dims=[3])

    p2 = torch.sigmoid(
        model(torch.flip(img_t, dims=[2]))
    ).flip(dims=[2])

    return ((p0 + p1 + p2) / 3.0).cpu()

@torch.no_grad()
def evaluate_model(model: nn.Module,
                   image_paths,
                   mask_paths,
                   img_size: int,
                   device: str,
                   threshold: float = 0.5,
                   use_tta: bool = True,
                   data_loader: Optional[DataLoader] = None,
                   ) -> tuple:
    """
    Returns:
        dice_list, iou_list, sens_list, spec_list, prec_list, f1_list

    If ``data_loader`` is provided, runs batched inference on (image, mask) pairs
    already resized to ``img_size`` (fast path for val/test). Otherwise iterates
    ``image_paths`` / ``mask_paths`` and resizes predictions to each image's
    native resolution (slower; use for parity with threshold tuning caches).
    """
    model.eval()
    val_tf = get_val_transform(img_size)
    pin = torch.cuda.is_available()

    dice_list, iou_list = [], []
    sens_list, spec_list, prec_list, f1_list = [], [], [], []

    if data_loader is not None:
        for imgs, masks in tqdm(
            data_loader,
            total=len(data_loader),
            desc="Evaluating",
            leave=False,
        ):
            imgs = imgs.to(device, non_blocking=pin)
            if use_tta:
                prob_b = predict_tta(model, imgs, device).numpy()
            else:
                prob_b = torch.sigmoid(model(imgs)).cpu().numpy()
            masks_np = (masks.squeeze(1).numpy() > 0.5).astype(np.uint8)
            for b in range(prob_b.shape[0]):
                pred = (prob_b[b] > threshold).astype(np.uint8)
                gt = masks_np[b]
                d, i, se, sp, pr, f1 = compute_binary_seg_metrics(pred, gt)
                dice_list.append(d)
                iou_list.append(i)
                sens_list.append(se)
                spec_list.append(sp)
                prec_list.append(pr)
                f1_list.append(f1)
        return dice_list, iou_list, sens_list, spec_list, prec_list, f1_list

    for img_p, mask_p in tqdm(
        zip(image_paths, mask_paths),
        total=len(image_paths),
        desc="Evaluating",
        leave=False
    ):
        # --- Load image ---
        img_np = np.array(Image.open(img_p).convert("RGB"))
        orig_h, orig_w = img_np.shape[:2]

        img_t = val_tf(image=img_np)["image"].float().unsqueeze(0)

        # --- Inference ---
        if use_tta:
            prob = predict_tta(model, img_t, device).squeeze().numpy()
        else:
            prob = torch.sigmoid(
                model(img_t.to(device, non_blocking=pin))
            ).squeeze().cpu().numpy()

        # --- Resize back ---
        prob = cv2.resize(prob, (orig_w, orig_h),
                          interpolation=cv2.INTER_LINEAR)

        pred = (prob > threshold).astype(np.uint8)

        # --- Load GT ---
        gt = np.array(Image.open(mask_p).convert("L"))
        gt = (gt > 127).astype(np.uint8)

        # --- Metrics ---
        d, i, se, sp, pr, f1 = compute_binary_seg_metrics(pred, gt)
        dice_list.append(d)
        iou_list.append(i)
        sens_list.append(se)
        spec_list.append(sp)
        prec_list.append(pr)
        f1_list.append(f1)

    return dice_list, iou_list, sens_list, spec_list, prec_list, f1_list


print("Metrics and TTA helpers defined.")


In [ ]:
# ── 3.6  Training loop ────────────────────────────────────────────────────────

class ModelEMA:
    """Exponential moving average of model weights (evaluates more stable logits)."""

    def __init__(self, model: nn.Module, decay: float = 0.999):
        import copy
        self.ema = copy.deepcopy(model)
        self.decay = float(decay)
        self.ema.eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        d = self.decay
        for e, p in zip(self.ema.parameters(), model.parameters()):
            e.mul_(d).add_(p, alpha=1.0 - d)


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer,
    scheduler,
    n_epochs: int,
    device: str,
    model_name: str,
    ckpt_path: Path,
    patience: int = 10,
    grad_accum: int = 2,
    grad_clip_norm: float = 0.0,
    loss_fn=None,
    use_aux_loss: bool = False,
    aux_weight: float = 0.4,
    ema_decay: float = 0.0,
) -> list:
    """
    AMP-accelerated training with gradient accumulation and early stopping.

    grad_accum : effective batch = batch_size × grad_accum
    patience   : early-stop if val Dice does not improve for this many epochs
    loss_fn    : defaults to dice_bce_loss (Dice + BCEWithLogits)
    ema_decay  : if > 0, keeps an EMA copy for validation & checkpointing
    """
    if loss_fn is None:
        loss_fn = dice_bce_loss

    scaler    = GradScaler()
    best_dice = 0.0
    no_improv = 0
    log       = []
    ema       = ModelEMA(model, decay=ema_decay) if ema_decay > 0 else None
    pin       = device == "cuda"
    # Square input size from dataset (avoids relying on last-batch tensor shape)
    val_img_size = int(train_loader.dataset[0][0].shape[-1])

    for epoch in range(1, n_epochs + 1):
        if hasattr(model, "configure_epoch"):
            model.configure_epoch(epoch)

        model.train()
        run_loss = 0.0
        optimizer.zero_grad()

        for step, (imgs, masks) in enumerate(tqdm(train_loader,
                                                   desc=f"[{model_name}] Epoch {epoch}/{n_epochs}",
                                                   leave=False)):
            imgs = imgs.to(device, non_blocking=pin)
            masks = masks.to(device, non_blocking=pin)
            with autocast():
                if use_aux_loss:
                    logits, aux = model(imgs, return_aux=True)
                    loss = loss_fn(logits, masks) + aux_weight * loss_fn(aux, masks)
                else:
                    logits = model(imgs)
                    loss = loss_fn(logits, masks)
                loss = loss / grad_accum

            scaler.scale(loss).backward()

            if (step + 1) % grad_accum == 0 or (step + 1) == len(train_loader):
                if grad_clip_norm > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                if ema is not None:
                    ema.update(model)

            run_loss += loss.item() * grad_accum

        scheduler.step()
        avg_loss = run_loss / len(train_loader)

        eval_net = ema.ema if ema is not None else model
        val_dices, *_rest = evaluate_model(
            eval_net, None, None,
            val_img_size, device,
            threshold=0.5, use_tta=False, data_loader=val_loader,
        )
        val_dice = float(np.mean(val_dices)) if val_dices else 0.0
        log.append({"epoch": epoch, "loss": avg_loss, "val_dice": val_dice})
        print(f"[{model_name}] epoch {epoch:3d}  loss={avg_loss:.4f}  val_dice={val_dice:.4f}")

        if val_dice > best_dice:
            best_dice = val_dice
            no_improv = 0
            state = ema.ema.state_dict() if ema is not None else model.state_dict()
            torch.save(state, ckpt_path)
        else:
            no_improv += 1
            if no_improv >= patience:
                print(f"  ↳ Early stopping at epoch {epoch} (patience={patience})")
                break

    try:
        sd = torch.load(ckpt_path, map_location=device, weights_only=True)
    except TypeError:
        sd = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(sd)
    print(f"[{model_name}] Best val Dice = {best_dice:.4f}  (weights restored)")
    return log


def run_training_sanity_checks():
    """Quick numerical / shape checks (safe to run before training)."""
    x = torch.randn(2, 1, 64, 64, device=DEVICE)
    y = (torch.rand(2, 1, 64, 64, device=DEVICE) > 0.7).float()
    loss = dice_bce_loss(x, y)
    assert torch.isfinite(loss), "dice_bce_loss produced non-finite value"
    dl = dice_loss(torch.sigmoid(x), y)
    assert torch.isfinite(dl), "dice_loss non-finite"
    if DEVICE == "cuda":
        with autocast():
            loss2 = dice_bce_loss(x, y)
    else:
        loss2 = dice_bce_loss(x, y)
    assert torch.isfinite(loss2)
    print("Sanity OK: loss finite, AMP smoke test OK.")


run_training_sanity_checks()
print("Training loop defined.")


---
## §4 · Model Architectures

### Training flags — set to `False` to skip a model

```python
TRAIN_RESNET    = True   # ResNet-34 U-Net  (CNN baseline)
TRAIN_SEGFORMER = True   # SegFormer-B2     (efficient transformer)
```

Each model is defined as a unified wrapper: `model(pixel_values) → (B, 1, H, W)` raw logits.


In [ ]:
# ── Training flags ─────────────────────────────────────────────────────────────
# Set any flag to False to skip that model entirely (saves time & VRAM).
TRAIN_RESNET    = True   # ~24 M params  — fast CNN baseline
TRAIN_SEGFORMER = True   # ~27 M params  — efficient transformer


In [ ]:
# ── 4.1  Model 1 — ResNet-34 U-Net ────────────────────────────────────────────
#
# Architecture: ResNet-34 encoder with ImageNet-1k weights + symmetric U-Net decoder
#   • Skip connections from encoder stages to decoder → preserves fine-grained details
#   • Pre-training on ImageNet-1k provides strong low-level features (edges, textures)
#
# Hyperparameters:
#   IMG_SIZE  512  — matches dataset resolution; larger → more detail but 4× VRAM
#   BATCH      8   — fits T4 (16 GB); use 4 if OOM
#   EPOCHS    60   — sufficient with early stopping (patience=10)
#   LR      1e-4   — single flat LR; ResNet encoder is robust to this
#   GRAD_ACCUM 2   — effective batch = 16

RESNET_SIZE      = 512
RESNET_BATCH     = 8
RESNET_EPOCHS    = 60
RESNET_LR        = 1e-4
RESNET_GRAD_ACCUM = 2


def build_resnet34_unet():
    try:
        return smp.Unet(
            encoder_name="resnet34", encoder_weights="imagenet",
            in_channels=3, classes=1, activation=None,
        ).to(DEVICE)
    except Exception as e:
        print(f"ImageNet weights unavailable ({e}); using random init.")
        return smp.Unet(
            encoder_name="resnet34", encoder_weights=None,
            in_channels=3, classes=1, activation=None,
        ).to(DEVICE)


if TRAIN_RESNET:
    resnet_model = build_resnet34_unet()
    with torch.no_grad():
        _d = torch.randn(1, 3, RESNET_SIZE, RESNET_SIZE).to(DEVICE)
        _o = resnet_model(_d)
        assert _o.shape == (1, 1, RESNET_SIZE, RESNET_SIZE), f"Shape: {_o.shape}"
        del _d, _o
    n_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
    print(f"ResNet-34 U-Net  |  input {RESNET_SIZE}²  |  params {n_params/1e6:.1f}M  ✓")
else:
    resnet_model = None
    print("TRAIN_RESNET=False — ResNet-34 U-Net skipped.")


In [ ]:
# ── 4.2  Model 2 — SegFormer-B2 ───────────────────────────────────────────────
#
# Architecture: MiT-B2 hierarchical ViT encoder + lightweight all-MLP decoder
#   pixel_values (B, 3, 512, 512)
#   → MiT-B2: 4 stages of mix-depth self-attention (multi-scale features)
#   → All-MLP decoder (<1 M params) fuses multi-scale features
#   → logits (B, 1, 128, 128) → bilinear upsample → (B, 1, 512, 512)
#
# Pre-training: ImageNet-1k supervised (nvidia/mit-b2 on HuggingFace)
#
# Hyperparameters:
#   Differential LR: encoder=6e-5 (already trained), decoder=5e-4 (random init)
#   Weight decay 0.05: AdamW L2 standard for ViT fine-tuning
#   Warmup 5 epochs: ViT training is sensitive to LR at the start
#   GRAD_ACCUM 2: effective batch = 16

SEGFORMER_SIZE     = 512
SEGFORMER_BATCH    = 8
SEGFORMER_EPOCHS   = 60
SEGFORMER_WARMUP   = 5
SEGFORMER_LR_ENC   = 6e-5   # encoder — pre-trained weights
SEGFORMER_LR_DEC   = 5e-4   # decoder — random init; learns faster
SEGFORMER_WD       = 0.05
SEGFORMER_GRAD_ACCUM = 2
SEGFORMER_HF_NAME  = "nvidia/mit-b2"


class SegFormerWrapper(nn.Module):
    """HuggingFace SegFormer-B2 wrapped to return (B, 1, H, W) logits."""
    def __init__(self, hf_name: str = SEGFORMER_HF_NAME, local_files_only: bool = False):
        super().__init__()
        cfg = SegformerConfig.from_pretrained(hf_name)
        cfg.num_labels = 1
        self.segformer = SegformerForSemanticSegmentation.from_pretrained(
            hf_name, config=cfg, ignore_mismatched_sizes=True,
            local_files_only=local_files_only,
        )

    def forward(self, pixel_values: torch.Tensor, output_attentions: bool = False,
                **kwargs) -> torch.Tensor:
        B, C, H, W = pixel_values.shape
        out    = self.segformer(pixel_values, output_attentions=output_attentions, **kwargs)
        logits = out.logits                            # (B, 1, H//4, W//4)
        return F.interpolate(logits, size=(H, W),
                             mode="bilinear", align_corners=False)


if TRAIN_SEGFORMER:
    import time
    try:
        segformer_model = SegFormerWrapper().to(DEVICE)
    except Exception as e:
        print(f"SegFormer first attempt failed ({e}). Waiting 5s, retrying from local HF cache only...")
        time.sleep(5)
        try:
            segformer_model = SegFormerWrapper(local_files_only=True).to(DEVICE)
        except Exception as e2:
            raise RuntimeError(
                "SegFormer load failed twice. Check network/HF token or pre-download weights "
                f"(e.g. huggingface-cli download {SEGFORMER_HF_NAME}). Original error: {e!r}; "
                f"retry: {e2!r}"
            ) from e2

    with torch.no_grad():
        _d = torch.randn(1, 3, SEGFORMER_SIZE, SEGFORMER_SIZE).to(DEVICE)
        _o = segformer_model(_d)
        assert _o.shape == (1, 1, SEGFORMER_SIZE, SEGFORMER_SIZE), f"Shape: {_o.shape}"
        del _d, _o
    n_params = sum(p.numel() for p in segformer_model.parameters() if p.requires_grad)
    print(f"SegFormer-B2  |  input {SEGFORMER_SIZE}²  |  params {n_params/1e6:.1f}M  ✓")
else:
    segformer_model = None
    print("TRAIN_SEGFORMER=False — SegFormer-B2 skipped.")


In [ ]:
# This pipeline trains and evaluates only ResNet-34 U-Net and SegFormer-B2.


In [ ]:
# ── 4.4  Architecture summary ─────────────────────────────────────────────────
print("=" * 72)
print("  ARCHITECTURE SUMMARY")
print("=" * 72)
print(f"  {'Model':<30} {'Input':>6}  {'Params':>10}  {'Pre-training':<30}")
print(f"  {'-'*30}  {'-'*6}  {'-'*10}  {'-'*30}")

model_registry = []
for name, model, sz, pretrain, flag in [
    ("ResNet-34 U-Net",          resnet_model,    RESNET_SIZE,    "ImageNet-1k (supervised)",   TRAIN_RESNET),
    ("SegFormer-B2 (MiT-B2)",    segformer_model, SEGFORMER_SIZE, "ImageNet-1k (supervised)",   TRAIN_SEGFORMER),
]:
    status = "✓ active" if flag else "— skipped"
    params = f"{sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M" if model else "—"
    print(f"  {name:<30} {sz:>6}²  {params:>10}  {pretrain:<30}  {status}")
    if flag and model:
        model_registry.append((name, model, sz, pretrain))

print("=" * 72)


---
## Training (Section 5)

Each model is trained independently using the shared `train_model` loop.  
Only models with their flag set to `True` are trained.

### Shared training recipe

| Component | Value | Rationale |
|-----------|-------|-----------|
| Loss | Dice + BCEWithLogits (`dice_bce_loss`; optional Focal); `combined_loss` = Dice + Focal | Metric alignment + stable training; Focal if imbalance is harsh |
| Gradient accumulation | ×2 (ResNet/SegFormer) | Stable optimization with available GPU memory |
| LR schedule | Warmup (5 epochs default) → cosine to 1e-7 | Smoother optimisation at training start |
| AMP | float16 | ~2× speed, ~50% VRAM savings |
| Early stopping | patience=10 | Avoids overfitting on small DFU dataset |
| Augmentation | Elastic, shift/scale/rotate, CLAHE, GaussNoise, CoarseDropout | Stronger geom + photometric aug |


In [ ]:
# ── 5.1  Train ResNet-34 U-Net ─────────────────────────────────────────────────
if TRAIN_RESNET:
    print("=" * 60)
    print("  Training ResNet-34 U-Net")
    print("=" * 60)

    resnet_tl, resnet_vl, resnet_sl = make_loaders(RESNET_SIZE, RESNET_BATCH)

    resnet_optimizer = torch.optim.AdamW(
        resnet_model.parameters(), lr=RESNET_LR, weight_decay=1e-4
    )
    resnet_warmup   = LinearLR(resnet_optimizer, start_factor=0.01, end_factor=1.0,
                                total_iters=5)
    resnet_cosine   = CosineAnnealingLR(resnet_optimizer,
                                         T_max=max(RESNET_EPOCHS-5, 1), eta_min=1e-7)
    resnet_scheduler = SequentialLR(resnet_optimizer,
                                    schedulers=[resnet_warmup, resnet_cosine],
                                    milestones=[5])

    resnet_log = train_model(
        model=resnet_model,
        train_loader=resnet_tl,
        val_loader=resnet_vl,
        optimizer=resnet_optimizer,
        scheduler=resnet_scheduler,
        n_epochs=RESNET_EPOCHS,
        device=DEVICE,
        model_name="ResNet-34 U-Net",
        ckpt_path=CKPT_DIR / "resnet34_unet_best.pth",
        patience=10,
        grad_accum=RESNET_GRAD_ACCUM,
    )
else:
    resnet_log = []
    resnet_tl = resnet_vl = resnet_sl = None
    print("TRAIN_RESNET=False — skipping ResNet-34 training.")


In [ ]:
# ── 5.2  Train SegFormer-B2 ───────────────────────────────────────────────────
if TRAIN_SEGFORMER:
    print("=" * 60)
    print("  Training SegFormer-B2")
    print("=" * 60)

    segformer_tl, segformer_vl, segformer_sl = make_loaders(SEGFORMER_SIZE, SEGFORMER_BATCH)

    # Differential LR: lower for pre-trained encoder, higher for random-init decoder
    segformer_optimizer = torch.optim.AdamW([
        {"params": segformer_model.segformer.segformer.parameters(),
         "lr": SEGFORMER_LR_ENC},
        {"params": segformer_model.segformer.decode_head.parameters(),
         "lr": SEGFORMER_LR_DEC},
    ], weight_decay=SEGFORMER_WD)

    segformer_warmup   = LinearLR(segformer_optimizer, start_factor=0.01,
                                   end_factor=1.0, total_iters=SEGFORMER_WARMUP)
    segformer_cosine   = CosineAnnealingLR(segformer_optimizer,
                                            T_max=max(SEGFORMER_EPOCHS-SEGFORMER_WARMUP, 1),
                                            eta_min=1e-7)
    segformer_scheduler = SequentialLR(segformer_optimizer,
                                       schedulers=[segformer_warmup, segformer_cosine],
                                       milestones=[SEGFORMER_WARMUP])

    segformer_log = train_model(
        model=segformer_model,
        train_loader=segformer_tl,
        val_loader=segformer_vl,
        optimizer=segformer_optimizer,
        scheduler=segformer_scheduler,
        n_epochs=SEGFORMER_EPOCHS,
        device=DEVICE,
        model_name="SegFormer-B2",
        ckpt_path=CKPT_DIR / "segformer_b2_best.pth",
        patience=10,
        grad_accum=SEGFORMER_GRAD_ACCUM,
    )
else:
    segformer_log = []
    segformer_tl = segformer_vl = segformer_sl = None
    print("TRAIN_SEGFORMER=False — skipping SegFormer-B2 training.")


In [ ]:
# No additional model training block.


In [ ]:
# ── 5.4  Training curves ──────────────────────────────────────────────────────
logs_to_plot = []
if TRAIN_RESNET    and resnet_log:    logs_to_plot.append(("ResNet-34 U-Net",           resnet_log,    "#2E6DA4"))
if TRAIN_SEGFORMER and segformer_log: logs_to_plot.append(("SegFormer-B2",              segformer_log, "#C0570D"))

if logs_to_plot:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, log, color in logs_to_plot:
        epochs    = [e["epoch"]    for e in log]
        losses    = [e["loss"]     for e in log]
        val_dices = [e["val_dice"] for e in log]
        axes[0].plot(epochs, losses,    color=color, label=name)
        axes[1].plot(epochs, val_dices, color=color, label=name)

    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Train Loss")
    axes[0].set_title("Training loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val Dice (no TTA)")
    axes[1].set_title("Validation Dice"); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle("Training curves — all models", fontsize=13)
    plt.tight_layout()
    plt.savefig(VIZ_DIR / "training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("No training logs to plot.")


---
## §6 · Evaluation + TTA + Threshold Tuning

In [ ]:
# ── 6.1  Evaluate all active models with TTA ──────────────────────────────────


def _make_eval_loader(image_paths, mask_paths, img_size: int, batch_size: int = 4):
    ds = FUSDataset(image_paths, mask_paths, img_size, augment=False)
    nw = min(4, max(1, (os.cpu_count() or 2) - 1))
    pin = torch.cuda.is_available()
    kw = dict(num_workers=nw, pin_memory=pin, persistent_workers=(nw > 0))
    if nw > 0:
        kw["prefetch_factor"] = 2
    return DataLoader(ds, batch_size=batch_size, shuffle=False, **kw)


def evaluate_current_weights(model, name, img_size):
    print(f"Evaluating {name} (TTA, batched)...")
    ev_loader = _make_eval_loader(val_imgs, val_masks, img_size, batch_size=4)
    dl, il, sl, spl, pl, f1l = evaluate_model(
        model, None, None, img_size, DEVICE,
        threshold=0.5, use_tta=True, data_loader=ev_loader,
    )
    print(
        f"  {name}: Dice={np.nanmean(dl):.4f}  IoU={np.nanmean(il):.4f}  "
        f"Sens={np.nanmean(sl):.4f}  Spec={np.nanmean(spl):.4f}  "
        f"Prec={np.nanmean(pl):.4f}  F1={np.nanmean(f1l):.4f}"
    )
    return (
        float(np.nanmean(dl)), float(np.nanmean(il)),
        float(np.nanmean(sl)), float(np.nanmean(spl)),
        float(np.nanmean(pl)), float(np.nanmean(f1l)),
        dl, il, sl, spl, pl, f1l,
    )


MODEL_RESULTS = []

if TRAIN_RESNET and resnet_model:
    rd, ri, rs, rsp, rp, rf1, rdl, ril, rsl, rspl, rpl, rf1l = evaluate_current_weights(
        resnet_model, "ResNet-34 U-Net", RESNET_SIZE,
    )
    MODEL_RESULTS.append(
        ("ResNet-34 U-Net", rd, ri, rs, rsp, rp, rf1, rdl, ril, rsl, rspl, rpl, rf1l,
         resnet_model, RESNET_SIZE)
    )

if TRAIN_SEGFORMER and segformer_model:
    sd, si, ss, ssp, sp, sf1, sdl, sil, ssl, sspl, spl, sf1l = evaluate_current_weights(
        segformer_model, "SegFormer-B2", SEGFORMER_SIZE,
    )
    MODEL_RESULTS.append(
        ("SegFormer-B2", sd, si, ss, ssp, sp, sf1, sdl, sil, ssl, sspl, spl, sf1l,
         segformer_model, SEGFORMER_SIZE)
    )

assert MODEL_RESULTS, "No models were trained — set at least one TRAIN_* flag to True."

best_idx = int(np.argmax([r[1] for r in MODEL_RESULTS]))
(
    BEST_NAME, BEST_DICE, BEST_IOU,
    BEST_SENS, BEST_SPEC, BEST_PREC, BEST_F1,
    BEST_DL, BEST_IL, BEST_SL, BEST_SPL, BEST_PL, BEST_F1L,
    BEST_MODEL, BEST_SIZE,
) = MODEL_RESULTS[best_idx]

print(f"\nSelected model : {BEST_NAME}")
print(
    f"  Val Dice : {BEST_DICE:.4f}  |  Val IoU : {BEST_IOU:.4f}  |  "
    f"Sens : {BEST_SENS:.4f}  |  Spec : {BEST_SPEC:.4f}  |  "
    f"Prec : {BEST_PREC:.4f}  |  F1 : {BEST_F1:.4f}"
)


In [ ]:
# ── 6.2  Comparison plots ─────────────────────────────────────────────────────
names_short = [r[0] for r in MODEL_RESULTS]
dices  = [r[1] for r in MODEL_RESULTS]
ious   = [r[2] for r in MODEL_RESULTS]
colors = ["#C0570D" if i == best_idx else "#2E6DA4" for i in range(len(MODEL_RESULTS))]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(names_short, dices, color=colors, edgecolor="white")
for i, d in enumerate(dices):
    axes[0].text(i, d+0.005, f"{d:.3f}", ha="center", fontsize=9)
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("Val Dice")
axes[0].set_title("Dice comparison (orange = selected)")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(names_short, ious, color=colors, edgecolor="white")
for i, v in enumerate(ious):
    axes[1].text(i, v+0.005, f"{v:.3f}", ha="center", fontsize=9)
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("Val IoU")
axes[1].set_title("IoU comparison"); axes[1].tick_params(axis="x", rotation=15)

axes[2].scatter(BEST_IL, BEST_DL, alpha=0.4, s=12, color="#1B3A5C")
axes[2].set_xlabel("IoU"); axes[2].set_ylabel("Dice")
axes[2].set_title(f"{BEST_NAME} — per-image Dice vs IoU")
axes[2].axline((0, 0), slope=1, color="gray", linestyle="--", alpha=0.4)

plt.suptitle("Model comparison — Foot Ulcer Segmentation Challenge (val)", fontsize=13)
plt.tight_layout()
plt.savefig(VIZ_DIR / "model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── 6.3  Threshold tuning on val set ──────────────────────────────────────────
print("Tuning prediction threshold on validation set...")

val_tf = get_val_transform(BEST_SIZE)
probs_cache, gt_cache = [], []

for img_p, mask_p in tqdm(zip(val_imgs, val_masks),
                           total=len(val_imgs), desc="Caching val predictions"):
    img_np         = np.array(Image.open(img_p).convert("RGB"))
    orig_h, orig_w = img_np.shape[:2]
    img_t          = val_tf(image=img_np)["image"].float().unsqueeze(0)
    prob = predict_tta(BEST_MODEL, img_t, DEVICE).squeeze().numpy()
    prob = cv2.resize(prob, (orig_w, orig_h))
    probs_cache.append(prob)
    gt = (np.array(Image.open(mask_p).convert("L")) > 127).astype(np.uint8)
    gt_cache.append(gt)

thresholds = np.arange(0.30, 0.66, 0.05)
thresh_dices = {}

for thresh in thresholds:
    dices_t = []
    for prob, gt in zip(probs_cache, gt_cache):
        pred = (prob > thresh).astype(np.uint8)
        d, _ = compute_dice_iou(pred, gt)
        dices_t.append(d)
    thresh_dices[thresh] = float(np.mean(dices_t))

BEST_THRESHOLD = max(thresh_dices, key=thresh_dices.get)
print(f"\nThreshold sweep:")
for t, d in sorted(thresh_dices.items()):
    marker = "  ← selected" if abs(t - BEST_THRESHOLD) < 1e-6 else ""
    print(f"  thresh={t:.2f}  Dice={d:.4f}{marker}")

# Threshold curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(thresh_dices.keys()), list(thresh_dices.values()), "o-", color="#2E6DA4")
ax.axvline(BEST_THRESHOLD, color="#C0570D", linestyle="--",
           label=f"Best={BEST_THRESHOLD:.2f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Mean Val Dice")
ax.set_title(f"{BEST_NAME} — Threshold tuning")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(VIZ_DIR / "threshold_tuning.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── 6.4  Re-evaluate with optimal threshold + test set ────────────────────────
val_ev_loader = _make_eval_loader(val_imgs, val_masks, BEST_SIZE, batch_size=4)
(
    dice_l_opt, iou_l_opt, sens_l_opt, spec_l_opt, prec_l_opt, f1_l_opt,
) = evaluate_model(
    BEST_MODEL, None, None, BEST_SIZE, DEVICE,
    threshold=BEST_THRESHOLD, use_tta=True, data_loader=val_ev_loader,
)
BEST_DICE = float(np.nanmean(dice_l_opt))
BEST_IOU  = float(np.nanmean(iou_l_opt))
BEST_SENS = float(np.nanmean(sens_l_opt))
BEST_SPEC = float(np.nanmean(spec_l_opt))
BEST_PREC = float(np.nanmean(prec_l_opt))
BEST_F1   = float(np.nanmean(f1_l_opt))
BEST_DL, BEST_IL = dice_l_opt, iou_l_opt
BEST_SL, BEST_SPL, BEST_PL, BEST_F1L = sens_l_opt, spec_l_opt, prec_l_opt, f1_l_opt

print(f"[{BEST_NAME}] TTA + threshold={BEST_THRESHOLD:.2f}")
print(f"  Val Dice : {BEST_DICE:.4f}  |  Val IoU  : {BEST_IOU:.4f}")
print(
    f"  Val Sens : {BEST_SENS:.4f}  |  Val Spec : {BEST_SPEC:.4f}  |  "
    f"Val Prec : {BEST_PREC:.4f}  |  Val F1 : {BEST_F1:.4f}"
)

# Test set — sealed: default OFF; with RUN_TEST_ONCE=True, runs at most once per workspace
# unless you delete OUTPUT_DIR / "test_evaluated.lock".
RUN_TEST_ONCE = False   # ← set True only after all tuning is finalised
TEST_EVAL_LOCK = OUTPUT_DIR / "test_evaluated.lock"

if RUN_TEST_ONCE:
    if TEST_EVAL_LOCK.exists():
        print(
            f"\nTest evaluation skipped: found lock file {TEST_EVAL_LOCK} "
            "(delete it to re-run test evaluation)."
        )
    else:
        print(f"\nEvaluating {BEST_NAME} on TEST set (threshold={BEST_THRESHOLD:.2f})...")
        test_ev_loader = _make_eval_loader(test_imgs, test_masks, BEST_SIZE, batch_size=4)
        t_dl, t_il, t_sl, t_spl, t_pl, t_f1l = evaluate_model(
            BEST_MODEL, None, None, BEST_SIZE, DEVICE,
            threshold=BEST_THRESHOLD, use_tta=True, data_loader=test_ev_loader,
        )
        print(
            f"TEST  Dice={np.nanmean(t_dl):.4f} ± {np.nanstd(t_dl):.4f}  "
            f"IoU={np.nanmean(t_il):.4f} ± {np.nanstd(t_il):.4f}"
        )
        print(
            f"      Sens={np.nanmean(t_sl):.4f} ± {np.nanstd(t_sl):.4f}  "
            f"Spec={np.nanmean(t_spl):.4f} ± {np.nanstd(t_spl):.4f}  "
            f"Prec={np.nanmean(t_pl):.4f} ± {np.nanstd(t_pl):.4f}  "
            f"F1={np.nanmean(t_f1l):.4f} ± {np.nanstd(t_f1l):.4f}"
        )
        TEST_EVAL_LOCK.touch()
else:
    print("Test set not evaluated (RUN_TEST_ONCE=False). Set True once when tuning is complete.")


---
## §7 · Inference & Qualitative Analysis

In [ ]:
# ── 7.1  Inference helpers ─────────────────────────────────────────────────────
TISSUE_COLOR = np.array([220, 50, 50], dtype=np.uint8)
# Nominal cm² per predicted foreground pixel — not physical without calibration / scale reference.
PIXEL_AREA_CM2_NOMINAL = 0.0001
_val_tf = get_val_transform(BEST_SIZE)


def predict_mask(img_bgr: np.ndarray) -> tuple:
    h, w    = img_bgr.shape[:2]
    img_np  = img_bgr[:, :, ::-1]
    img_t   = _val_tf(image=img_np)["image"].float().unsqueeze(0)
    prob    = predict_tta(BEST_MODEL, img_t, DEVICE).squeeze().numpy()
    pred_sm = (prob > BEST_THRESHOLD).astype(np.uint8)
    pred    = cv2.resize(pred_sm, (w, h), interpolation=cv2.INTER_NEAREST)
    vis           = img_np.copy().astype(np.float32)
    vis[pred > 0] = vis[pred > 0] * 0.5 + TISSUE_COLOR.astype(np.float32) * 0.5
    return pred, vis.astype(np.uint8)


def visualize_prediction(img_path, mask_path, alpha: float = 0.5) -> tuple:
    img_bgr = cv2.imread(str(img_path))
    img_rgb = img_bgr[:, :, ::-1].copy()
    h, w    = img_bgr.shape[:2]
    gt_arr  = np.array(Image.open(mask_path).convert("L"))
    gt_mask = cv2.resize(gt_arr, (w, h), interpolation=cv2.INTER_NEAREST)
    gt_ov   = img_rgb.copy().astype(np.float32)
    gt_ov[gt_mask > 127] = (gt_ov[gt_mask > 127] * (1-alpha)
                             + TISSUE_COLOR.astype(np.float32) * alpha)
    _, pred_vis = predict_mask(img_bgr)
    return img_rgb, gt_ov.astype(np.uint8), pred_vis

print(f"Inference helpers ready  (threshold={BEST_THRESHOLD:.2f}, TTA=on).")


In [ ]:
# ── 7.2  Qualitative results — random positive samples ────────────────────────
val_pos = [(i, m) for i, m in zip(val_imgs, val_masks)
           if (np.array(Image.open(m)) > 0).any()]
samples = random.sample(val_pos, min(5, len(val_pos)))

fig, axes = plt.subplots(len(samples), 3, figsize=(15, 5*len(samples)))
if len(samples) == 1: axes = axes[np.newaxis, :]
fig.suptitle(f"Inference — {BEST_NAME}  |  Original  |  Ground truth  |  Prediction",
             fontsize=12)

for row, (img_p, mask_p) in enumerate(samples):
    orig, gt_ov, pred_ov = visualize_prediction(img_p, mask_p)
    for col, (title, vis) in enumerate(
        zip(["Original", "Ground truth", f"{BEST_NAME} prediction"],
            [orig, gt_ov, pred_ov])
    ):
        axes[row, col].imshow(vis); axes[row, col].axis("off")
        if row == 0: axes[row, col].set_title(title, fontsize=11)

plt.tight_layout()
plt.savefig(VIZ_DIR / "inference_samples.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── 7.3  Failure cases — 4 lowest-Dice samples ────────────────────────────────
val_scored = sorted(zip(val_imgs, val_masks, BEST_DL), key=lambda x: x[2])
worst_4    = val_scored[:4]

fig, axes = plt.subplots(len(worst_4), 3, figsize=(15, 5*len(worst_4)))
if len(worst_4) == 1: axes = axes[np.newaxis, :]
fig.suptitle("Failure cases — 4 lowest Dice scores", fontsize=13)

for row, (img_p, mask_p, dice) in enumerate(worst_4):
    orig, gt_ov, pred_ov = visualize_prediction(img_p, mask_p)
    for col, vis in enumerate([orig, gt_ov, pred_ov]):
        axes[row, col].imshow(vis); axes[row, col].axis("off")
    axes[row, 2].set_title(f"Dice = {dice:.3f}", fontsize=10, color="red")

axes[0, 0].set_title("Original", fontsize=11)
axes[0, 1].set_title("Ground truth", fontsize=11)
axes[0, 2].set_title("Prediction", fontsize=11)

plt.tight_layout()
plt.savefig(VIZ_DIR / "failure_cases.png", dpi=120, bbox_inches="tight")
plt.show()


---
## §8 · XAI — Segmentation-Native Explainability

The original Grad-CAM implementation was not designed for segmentation — it backprops from a scalar (`.max()`) rather than the spatial prediction, making the explanation **classification-level**, not pixel-level.  
This section replaces it with **three segmentation-native methods**:

| Method | How it works | Why better for segmentation |
|--------|-------------|----------------------------|
| **Score-CAM** | Computes channel-wise contribution by forward-passing masked images; no gradient needed | Gradient-free → stable; attribution tied to actual mask score |
| **Attention Rollout** | Propagates self-attention weights across all transformer layers | Shows the actual receptive field the model used; works for SegFormer |
| **RISE** | Occlusion-based saliency using random binary masks | Model-agnostic; works for ResNet U-Net and transformers equally |

All three methods produce **spatial heatmaps aligned with the prediction mask**, not just a coarse class activation.


In [ ]:
# ── 8.1  XAI flag & imports ───────────────────────────────────────────────────
ENABLE_XAI = True    # ← set False to skip RISE (slow)

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.lib import colors as rl_colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Image as RLImage, Table, TableStyle,
)

print("XAI imports OK.")


In [ ]:
# ── 8.2  Score-CAM (segmentation-native, gradient-free) ───────────────────────
#
# Score-CAM algorithm for segmentation:
#   1. Extract feature maps from a target layer
#   2. For each channel: upsample the channel to input resolution, use it as a mask
#   3. Run forward pass with masked image → get segmentation score (mean logit over pred region)
#   4. Weight each channel by its score, sum → saliency map
#
# This ties the attribution directly to the segmentation output — not a classification scalar.

class ScoreCAMSegmentation:
    """
    Score-CAM adapted for binary segmentation models.
    Works for any model that returns (B, 1, H, W) logits.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module, device: str):
        self.model   = model.eval()
        self.device  = device
        self._fmaps  = []
        target_layer.register_forward_hook(
            lambda m, i, o: self._fmaps.append(
                (o[0] if isinstance(o, tuple) else o).detach()
            )
        )

    def __call__(self, img_bgr: np.ndarray, n_channels: int = 64) -> tuple:
        """
        n_channels: number of channels to sample (top-n by L2 norm for speed).
        Returns (saliency_map [H, W], overlay [H, W, 3]).
        """
        h, w    = img_bgr.shape[:2]
        img_rgb = img_bgr[:, :, ::-1]
        img_t   = _val_tf(image=img_rgb)["image"].float().unsqueeze(0).to(self.device)
        img_f32 = img_rgb.astype(np.float32) / 255.0

        # Base forward — collect feature maps
        self._fmaps.clear()
        with torch.no_grad():
            base_logits = self.model(img_t)   # (1, 1, H, W)
        base_prob = torch.sigmoid(base_logits)
        pred_mask = (base_prob.squeeze().cpu().numpy() > 0.5).astype(np.float32)

        if not self._fmaps:
            saliency = np.zeros((h, w), dtype=np.float32)
            return saliency, img_rgb.copy()

        fmap = self._fmaps[-1]   # (1 or B, C, H', W') or (B, T, C) for ViT
        if fmap.dim() == 3:
            # ViT: (B, T, C) → reshape to spatial
            B, T, C = fmap.shape
            G = int(T**0.5)
            if G*G == T:
                fmap = fmap.permute(0, 2, 1).reshape(B, C, G, G)
            else:
                saliency = np.zeros((h, w), dtype=np.float32)
                return saliency, img_rgb.copy()

        _, C, Hf, Wf = fmap.shape

        # Select top-n channels by activation norm
        channel_norms = fmap[0].view(C, -1).norm(dim=1)
        top_channels  = channel_norms.topk(min(n_channels, C)).indices

        saliency  = np.zeros((h, w), dtype=np.float32)
        norm_mean = torch.tensor(IMGNET_MEAN, device=self.device).view(1, 3, 1, 1)
        norm_std  = torch.tensor(IMGNET_STD,  device=self.device).view(1, 3, 1, 1)

        for ch_idx in top_channels:
            ch_map = fmap[0, ch_idx].cpu().numpy().astype(np.float32)
            # Normalise channel map to [0, 1]
            ch_min, ch_max = ch_map.min(), ch_map.max()
            if ch_max - ch_min < 1e-6:
                continue
            ch_norm = (ch_map - ch_min) / (ch_max - ch_min + 1e-8)
            # Upsample to input resolution
            ch_up = cv2.resize(ch_norm, (w, h))

            # Mask input image with this channel
            masked = img_t * torch.tensor(ch_up, device=self.device,
                                           dtype=torch.float32).unsqueeze(0).unsqueeze(0)

            with torch.no_grad():
                logits_masked = self.model(masked)
            prob_masked = torch.sigmoid(logits_masked).squeeze().cpu().numpy()

            # Score = mean probability in the predicted region
            if pred_mask.sum() > 0:
                score = float(prob_masked[pred_mask > 0].mean())
            else:
                score = float(prob_masked.mean())

            saliency += ch_up * score

        if saliency.max() > 0:
            saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

        heat_u8  = (saliency * 255).astype(np.uint8)
        heat_rgb = cv2.applyColorMap(heat_u8, cv2.COLORMAP_INFERNO)[:, :, ::-1]
        overlay  = (img_f32 * 255 * 0.5 + heat_rgb * 0.5).astype(np.uint8)
        return saliency, overlay


def _get_score_cam_layer(model: nn.Module, model_name: str):
    """Return the target layer for Score-CAM based on model type."""    
    if "ResNet" in model_name or "resnet" in model_name.lower():
        return model.encoder.layer4[-1]
    elif "SegFormer" in model_name:
        return model.segformer.segformer.encoder.block[-1][-1].layer_norm_2
    else:
        raise ValueError(f"Unknown model type: {model_name}")


_score_cam_layer = _get_score_cam_layer(BEST_MODEL, BEST_NAME)
score_cam_engine = ScoreCAMSegmentation(BEST_MODEL, _score_cam_layer, DEVICE)
print(f"Score-CAM engine ready for {BEST_NAME}.")


In [ ]:
# ── 8.3  Attention Rollout ────────────────────────────────────────────────────
#
# Attention Rollout (Abnar & Zuidema, 2020):
#   Propagates attention weights through all transformer layers via matrix multiplication.
#   Each layer's attention matrix is mixed with an identity (residual connection).
#   The product of all layer-wise attentions gives the effective attention from input
#   patches to the CLS token — a proxy for "which image regions drove the prediction".
#
# For SegFormer, this produces a spatial heatmap at patch resolution,
# which is then upsampled to the full image size.
# For ResNet U-Net, we return a uniform map (attention rollout requires self-attention).
#
# Note: HuggingFace SegFormer must run with output_attentions=True or attentions are None.

class AttentionRollout:
    """
    Attention Rollout for transformer-based segmentation models.
    Accumulates attention maps across all transformer blocks.
    """

    def __init__(self, model: nn.Module, model_name: str):
        self.model      = model.eval()
        self.model_name = model_name
        self.attentions = []
        self._register_hooks()

    def _register_hooks(self):
        if "SegFormer" in self.model_name:
            for stage in self.model.segformer.segformer.encoder.block:
                for block in stage:
                    block.attention.self.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        if isinstance(out, tuple) and len(out) >= 2 and out[1] is not None:
            self.attentions.append(out[1].detach().cpu())
        elif isinstance(out, torch.Tensor) and out.dim() == 4:
            # Some implementations return the attention weights directly
            self.attentions.append(out.detach().cpu())

    def get(self, img_bgr: np.ndarray) -> np.ndarray:
        h, w = img_bgr.shape[:2]

        if "ResNet" in self.model_name:
            return np.zeros((h, w), dtype=np.float32)

        img_np = img_bgr[:, :, ::-1]
        img_t  = _val_tf(image=img_np)["image"].float().unsqueeze(0)
        self.attentions.clear()

        with torch.no_grad():
            if "SegFormer" in self.model_name:
                self.model(img_t.to(DEVICE), output_attentions=True)
            else:
                self.model(img_t.to(DEVICE))

        if not self.attentions:
            return np.zeros((h, w), dtype=np.float32)

        # Rollout: multiply attention matrices layer by layer
        rollout = None
        for attn in self.attentions:
            # attn: (B, heads, T, T) — average over heads
            if attn.dim() == 4:
                attn_avg = attn.mean(dim=1)   # (B, T, T)
            else:
                attn_avg = attn

            # Mix with identity for residual connection
            I = torch.eye(attn_avg.shape[-1]).unsqueeze(0)
            attn_res = 0.5 * attn_avg + 0.5 * I

            # Row-normalise
            attn_res = attn_res / attn_res.sum(dim=-1, keepdim=True).clamp(min=1e-8)

            rollout = attn_res if rollout is None else torch.bmm(rollout, attn_res)

        # Take the first row (CLS→patches) or mean across tokens
        rollout_np = rollout[0].numpy()
        if rollout_np.shape[0] > 1:
            amap = rollout_np[0, 1:]   # CLS → patch tokens
        else:
            amap = rollout_np[0]

        G = int(len(amap)**0.5)
        if G*G < len(amap): G += 1
        amap_padded = np.zeros(G*G, dtype=np.float32)
        amap_padded[:len(amap)] = amap
        amap_2d = amap_padded[:G*G].reshape(G, G)
        amap_2d = (amap_2d - amap_2d.min()) / (amap_2d.max() - amap_2d.min() + 1e-8)
        return cv2.resize(amap_2d, (w, h))


attn_rollout = AttentionRollout(BEST_MODEL, BEST_NAME)
print(f"Attention Rollout engine ready for {BEST_NAME}.")


In [ ]:
# ── 8.4  RISE (model-agnostic occlusion saliency) ────────────────────────────
from typing import Optional

#
# RISE (Randomised Input Sampling for Explanation):
#   Generates N random binary masks, applies each to the input image,
#   runs forward pass and records the segmentation score.
#   The saliency map = weighted sum of masks, weights = per-mask scores.
#   Works for any model — no gradient or architecture assumptions needed.
#
# n_masks=300: more → smoother map but slower; 200–300 is a good balance

def rise_segmentation(
    img_bgr: np.ndarray,
    n_masks: int = 300,
    mask_prob: float = 0.5,
    mask_size: int = 8,
    model: Optional[nn.Module] = None,
    prob_threshold: Optional[float] = None,
    device: Optional[str] = None,
) -> np.ndarray:
    """
    RISE adapted for binary segmentation:
    score = mean predicted probability within the current best predicted region.
    """
    _model = model if model is not None else BEST_MODEL
    _thr = prob_threshold if prob_threshold is not None else BEST_THRESHOLD
    _dev = device if device is not None else DEVICE

    h, w   = img_bgr.shape[:2]
    grid_h = (h + mask_size - 1) // mask_size
    grid_w = (w + mask_size - 1) // mask_size

    # Pre-compute base prediction region
    img_np = img_bgr[:, :, ::-1]
    img_t  = _val_tf(image=img_np)["image"].float().unsqueeze(0)
    with torch.no_grad():
        base_prob = torch.sigmoid(_model(img_t.to(_dev))).squeeze().cpu().numpy()
    pred_region = base_prob > _thr

    saliency    = np.zeros((h, w), dtype=np.float32)
    weights_sum = np.zeros((h, w), dtype=np.float32)

    for _ in tqdm(range(n_masks), desc="RISE", leave=False):
        grid       = (np.random.rand(grid_h, grid_w) < mask_prob).astype(np.float32)
        mask       = cv2.resize(grid, (w, h), interpolation=cv2.INTER_NEAREST)
        masked_bgr = (img_bgr.astype(np.float32) * mask[:,:,None]).astype(np.uint8)

        masked_np = masked_bgr[:, :, ::-1]
        masked_t  = _val_tf(image=masked_np)["image"].float().unsqueeze(0)
        with torch.no_grad():
            logit = _model(masked_t.to(_dev))
        prob_masked = torch.sigmoid(logit).squeeze().cpu().numpy()

        # Score = mean probability in predicted region (segmentation-specific)
        if pred_region.sum() > 0:
            score = float(prob_masked[pred_region].mean())
        else:
            score = float(prob_masked.mean())

        saliency    += mask * score
        weights_sum += mask

    with np.errstate(divide="ignore", invalid="ignore"):
        result = np.where(weights_sum > 0, saliency / weights_sum, 0.0)
    result = (result - result.min()) / (result.max() - result.min() + 1e-8)
    return result.astype(np.float32)


print("RISE (segmentation-native) defined.")


In [ ]:
# ── 8.5  Compute and save all XAI artefacts ───────────────────────────────────

def apply_heatmap(img_rgb, heatmap, colormap=cv2.COLORMAP_JET, alpha=0.5):
    heat_u8  = (heatmap * 255).astype(np.uint8)
    heat_rgb = cv2.applyColorMap(heat_u8, colormap)[:, :, ::-1]
    return (img_rgb*(1-alpha) + heat_rgb*alpha).astype(np.uint8)


def make_xai_panel(original, mask_ov, scam_ov, rollout_ov, rise_ov, title=""):
    panels = [original, mask_ov, scam_ov, rollout_ov, rise_ov]
    labels = ["Original", "Segmentation", "Score-CAM", "Attn Rollout", "RISE"]
    h      = max(p.shape[0] for p in panels)
    w_each = max(p.shape[1] for p in panels)
    canvas = np.ones((h+28, w_each*5+16, 3), dtype=np.uint8) * 240
    for i, (panel, label) in enumerate(zip(panels, labels)):
        ph, pw = panel.shape[:2]
        x_off  = i*(w_each+4)
        canvas[28:28+ph, x_off:x_off+pw] = panel
        cv2.putText(canvas, label, (x_off+4, 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (50,50,50), 1, cv2.LINE_AA)
    if title:
        cv2.putText(canvas, title, (4, canvas.shape[0]-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (80,80,80), 1, cv2.LINE_AA)
    return canvas


xai_samples = random.sample(
    [(i, m) for i, m in zip(val_imgs, val_masks)
     if (np.array(Image.open(m)) > 0).any()],
    k=min(8, sum(1 for m in val_masks
                 if (np.array(Image.open(m)) > 0).any()))
)
xai_records = []

for idx, (img_p, mask_p) in enumerate(tqdm(xai_samples, desc="XAI samples")):
    sid     = f"sample_{idx:03d}"
    img_bgr = cv2.imread(str(img_p))
    img_rgb = img_bgr[:, :, ::-1].copy()
    h, w    = img_bgr.shape[:2]
    img_f32 = img_rgb.astype(np.float32) / 255.0

    # Ground-truth overlay
    gt_raw  = np.array(Image.open(mask_p).convert("L"))
    gt_mask = cv2.resize(gt_raw, (w, h), interpolation=cv2.INTER_NEAREST)
    mask_ov = img_rgb.copy().astype(np.float32)
    mask_ov[gt_mask > 127] = (mask_ov[gt_mask > 127]*0.55
                               + TISSUE_COLOR.astype(np.float32)*0.45)
    mask_ov = mask_ov.astype(np.uint8)

    # Score-CAM
    try:
        scam_map, scam_ov = score_cam_engine(img_bgr, n_channels=64)
    except Exception as e:
        print(f"  Score-CAM failed for {sid}: {e}")
        scam_map = np.zeros((h, w), dtype=np.float32)
        scam_ov  = img_rgb.copy()

    # Attention Rollout
    try:
        rollout_map = attn_rollout.get(img_bgr)
        rollout_ov  = apply_heatmap(img_rgb, rollout_map, cv2.COLORMAP_HOT, 0.55)
    except Exception as e:
        print(f"  Attention Rollout failed for {sid}: {e}")
        rollout_map = np.zeros((h, w), dtype=np.float32)
        rollout_ov  = img_rgb.copy()

    # RISE
    if ENABLE_XAI:
        try:
            rise_map = rise_segmentation(
                img_bgr, n_masks=300,
                model=BEST_MODEL, prob_threshold=BEST_THRESHOLD, device=DEVICE,
            )
            rise_ov  = apply_heatmap(img_rgb, rise_map, cv2.COLORMAP_PLASMA, 0.55)
        except Exception as e:
            print(f"  RISE failed for {sid}: {e}")
            rise_map = scam_map
            rise_ov  = scam_ov.copy()
    else:
        rise_map = scam_map
        rise_ov  = apply_heatmap(img_rgb, scam_map, cv2.COLORMAP_PLASMA, 0.55)

    # Prediction metrics
    pred_bin, _ = predict_mask(img_bgr)
    gt_bin      = (gt_mask > 127).astype(np.uint8)
    dice, iou, sens, spec, prec, f1 = compute_binary_seg_metrics(pred_bin, gt_bin)
    ulcer_px    = int(pred_bin.sum())
    ulcer_cm2   = round(ulcer_px * PIXEL_AREA_CM2_NOMINAL, 2)

    img_t = _val_tf(image=img_rgb)["image"].float().unsqueeze(0)
    with torch.no_grad():
        logit = BEST_MODEL(img_t.to(DEVICE))
    confidence = float(torch.sigmoid(logit).max().cpu())

    # Save individual artefacts
    for fname, arr in [
        (f"{sid}_original.png",  cv2.cvtColor(img_rgb,   cv2.COLOR_RGB2BGR)),
        (f"{sid}_mask.png",      cv2.cvtColor(mask_ov,   cv2.COLOR_RGB2BGR)),
        (f"{sid}_scorecam.png",  cv2.cvtColor(scam_ov,   cv2.COLOR_RGB2BGR)),
        (f"{sid}_rollout.png",   cv2.cvtColor(rollout_ov, cv2.COLOR_RGB2BGR)),
        (f"{sid}_rise.png",      cv2.cvtColor(rise_ov,   cv2.COLOR_RGB2BGR)),
    ]:
        cv2.imwrite(str(XAI_DIR / fname), arr)

    area_note = (
        f"Dice={dice:.3f} IoU={iou:.3f} "
        f"Ulcer≈{ulcer_cm2}cm² (nominal, uncalibrated)"
    )
    combined = make_xai_panel(img_rgb, mask_ov, scam_ov, rollout_ov, rise_ov, title=area_note)
    cv2.imwrite(str(XAI_DIR / f"{sid}_combined.png"),
                cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))

    xai_records.append({
        "sample_id": sid, "image_path": str(img_p),
        "model": BEST_NAME, "threshold": BEST_THRESHOLD,
        "dice": round(dice,4), "iou": round(iou,4),
        "sensitivity": None if np.isnan(sens) else round(float(sens), 4),
        "specificity": None if np.isnan(spec) else round(float(spec), 4),
        "precision": None if np.isnan(prec) else round(float(prec), 4),
        "f1": None if np.isnan(f1) else round(float(f1), 4),
        "confidence": round(confidence,4),
        "ulcer_pixels": ulcer_px,
        "ulcer_area_cm2_nominal": ulcer_cm2,
        "pixel_area_cm2_nominal": PIXEL_AREA_CM2_NOMINAL,
        "area_disclaimer": (
            "Ulcer area uses PIXEL_AREA_CM2_NOMINAL × predicted pixels; not a physical "
            "measurement without calibration or a reference scale in the image."
        ),
        "xai_files": {k: f"{sid}_{k}.png"
                      for k in ["original","mask","scorecam","rollout","rise","combined"]},
    })

print(f"\n✓ XAI artefacts saved for {len(xai_samples)} samples → {XAI_DIR}")


In [ ]:
# ── 8.6  Display combined panels ──────────────────────────────────────────────
n_show = min(4, len(xai_records))
fig, axes = plt.subplots(n_show, 1, figsize=(22, 6*n_show))
if n_show == 1: axes = [axes]

for ax, rec in zip(axes, xai_records[:n_show]):
    comb = cv2.cvtColor(cv2.imread(str(XAI_DIR / rec["xai_files"]["combined"])),
                        cv2.COLOR_BGR2RGB)
    ax.imshow(comb); ax.axis("off")
    ax.set_title(
        f"{rec['sample_id']} | {rec['model']} | "
        f"Dice={rec['dice']:.3f} | IoU={rec['iou']:.3f} | "
        f"Ulcer≈{rec['ulcer_area_cm2_nominal']} cm² (nominal)",
        fontsize=10
    )

plt.suptitle("XAI — Original | Segmentation | Score-CAM | Attention Rollout | RISE",
             fontsize=12)
plt.tight_layout()
plt.savefig(VIZ_DIR / "xai_panels.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
# ── 8.7  Per-sample clinical PDF reports ──────────────────────────────────────
def build_clinical_pdf(record: dict, out_path: Path) -> None:
    doc   = SimpleDocTemplate(str(out_path), pagesize=A4,
                               rightMargin=2*cm, leftMargin=2*cm,
                               topMargin=2*cm, bottomMargin=2*cm)
    ss    = getSampleStyleSheet()
    story = []

    story.append(Paragraph("<b>Glunova AI Platform — DFU Clinical Report</b>", ss["Title"]))
    story.append(Spacer(1, 0.3*cm))
    story.append(Paragraph(
        f"Generated: {datetime.datetime.now():%Y-%m-%d %H:%M}  |  "
        f"Sample: {record['sample_id']}  |  Model: {record['model']}  |  "
        f"Threshold: {record['threshold']:.2f}",
        ss["Normal"],
    ))
    story.append(Spacer(1, 0.5*cm))

    story.append(Paragraph("<b>Quantitative Metrics</b>", ss["Heading2"]))
    story.append(Spacer(1, 0.2*cm))
    severity = (
        "HIGH — refer immediately"
        if record["ulcer_area_cm2_nominal"] > 4 or record["confidence"] < 0.50
        else "MODERATE — monitor closely"
        if record["ulcer_area_cm2_nominal"] > 1
        else "LOW — routine follow-up"
    )

    def _fmt(v, nd=4):
        return "—" if v is None else f"{v:.{nd}f}"

    rows = [
        ["Metric", "Value", "Threshold / note"],
        ["Dice score",          f"{record['dice']:.4f}",          "> 0.65 (acceptable)"],
        ["IoU (Jaccard)",       f"{record['iou']:.4f}",           "> 0.50 (acceptable)"],
        ["Sensitivity (recall)", _fmt(record.get("sensitivity")), "Higher is better for screening (missed ulcer = FN)"],
        ["Specificity",          _fmt(record.get("specificity")), "True-negative rate"],
        ["Precision (PPV)",      _fmt(record.get("precision")),  "Positive predictive value"],
        ["F1 score",             _fmt(record.get("f1")),         "Harmonic mean of prec. & recall"],
        ["Model confidence",    f"{record['confidence']:.4f}",    "> 0.70 (high)"],
        ["Ulcer area (nominal)", f"{record['ulcer_area_cm2_nominal']} cm²", "Not physical — see disclaimer"],
        ["Severity",            severity,                         "—"],
        ["AI model",            record["model"],                  "—"],
    ]
    tbl = Table(rows, colWidths=[5*cm, 3.5*cm, 6.5*cm])
    tbl.setStyle(TableStyle([
        ("BACKGROUND",    (0,0),(-1,0), rl_colors.HexColor("#1B3A5C")),
        ("TEXTCOLOR",     (0,0),(-1,0), rl_colors.white),
        ("FONTNAME",      (0,0),(-1,0), "Helvetica-Bold"),
        ("ROWBACKGROUNDS",(0,1),(-1,-1),
         [rl_colors.HexColor("#EBF5FB"), rl_colors.white]),
        ("GRID",          (0,0),(-1,-1), 0.5, rl_colors.HexColor("#BDD7EE")),
        ("FONTSIZE",      (0,0),(-1,-1), 9),
        ("TOPPADDING",    (0,0),(-1,-1), 4),
        ("BOTTOMPADDING", (0,0),(-1,-1), 4),
    ]))
    story.append(tbl); story.append(Spacer(1, 0.5*cm))

    story.append(Paragraph(
        "<b>Area disclaimer:</b> "
        + record.get(
            "area_disclaimer",
            "Ulcer area is a nominal heuristic, not a calibrated physical measurement.",
        ),
        ss["Normal"],
    ))
    story.append(Spacer(1, 0.4*cm))

    img_w = 8.5*cm
    story.append(Paragraph("<b>Segmentation Overlay</b>", ss["Heading2"]))
    story.append(Spacer(1, 0.2*cm))
    story.append(RLImage(str(XAI_DIR / record["xai_files"]["mask"]),
                          width=img_w, height=img_w*0.75))
    story.append(Spacer(1, 0.4*cm))

    story.append(Paragraph("<b>Score-CAM  |  Attention Rollout  |  RISE</b>", ss["Heading2"]))
    story.append(Spacer(1, 0.2*cm))
    trio = Table([[
        RLImage(str(XAI_DIR / record["xai_files"]["scorecam"]),
                width=5*cm, height=5*cm*0.75),
        RLImage(str(XAI_DIR / record["xai_files"]["rollout"]),
                width=5*cm, height=5*cm*0.75),
        RLImage(str(XAI_DIR / record["xai_files"]["rise"]),
                width=5*cm, height=5*cm*0.75),
    ]], colWidths=[5.3*cm, 5.3*cm, 5.3*cm])
    story.append(trio); story.append(Spacer(1, 0.4*cm))

    story.append(Paragraph("<b>XAI Method Explanations</b>", ss["Heading2"]))
    for note in [
        "<b>Score-CAM:</b> Gradient-free attribution — channels that most influence the segmentation score are highlighted. Spatially aligned with the predicted mask.",
        "<b>Attention Rollout:</b> Propagated self-attention across all transformer layers — shows which image patches the model attended to when forming its prediction.",
        "<b>RISE:</b> Occlusion-based saliency — regions where masking reduces segmentation confidence are highlighted. Works for all architectures (CNN and transformers).",
        "<b>Disclaimer:</b> AI-generated output. Must be reviewed by a qualified clinician before clinical use.",
    ]:
        story.append(Paragraph(note, ss["Normal"])); story.append(Spacer(1, 0.2*cm))

    doc.build(story)


for rec in tqdm(xai_records, desc="Generating PDF reports"):
    pdf_path = XAI_DIR / f"{rec['sample_id']}_report.pdf"
    build_clinical_pdf(rec, pdf_path)
    rec["pdf_path"] = str(pdf_path)

with open(XAI_DIR / "xai_index.json", "w") as f:
    json.dump(xai_records, f, indent=2)

print(f"\n✓ PDF reports and XAI index saved → {XAI_DIR}")


---
## §9 · Export

Single zip archive **`dfu_seg_export.zip`** — everything needed for Phase 2 inference.

```
dfu_seg_export.zip
├── {model_slug}_weights.pth        — state dict (no training state)
├── {model_slug}.onnx               — portable ONNX model (opset 17, dynamic axes)
├── {model_slug}_config.json        — preprocessing config + best val metrics
└── xai/                            — all XAI artefacts + PDF reports
    ├── sample_000_combined.png
    ├── sample_000_scorecam.png
    ├── ...
    ├── sample_NNN_report.pdf
    └── xai_index.json
```


In [ ]:
# ── 9.1  Save weights + config ────────────────────────────────────────────────
EXPORT_DIR = OUTPUT_DIR / "dfu_seg_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for f in EXPORT_DIR.iterdir():
    if f.is_file(): f.unlink()
    elif f.is_dir(): shutil.rmtree(f)

_slug_map = {
    "ResNet-34 U-Net":          "resnet34_unet",
    "SegFormer-B2":             "segformer_b2",
}
slug = _slug_map.get(BEST_NAME, "best_model")

weights_path = EXPORT_DIR / f"{slug}_weights.pth"
torch.save(BEST_MODEL.state_dict(), weights_path)

model_config = {
    "model_name":       BEST_NAME,
    "slug":             slug,
    "img_size":         BEST_SIZE,
    "num_classes":      1,
    "mean":             IMGNET_MEAN,
    "std":              IMGNET_STD,
    "threshold":        float(BEST_THRESHOLD),
    "tta_enabled":      True,
    "best_val_dice":    round(BEST_DICE, 4),
    "best_val_iou":     round(BEST_IOU, 4),
    "best_val_sensitivity":  round(BEST_SENS, 4),
    "best_val_specificity":    round(BEST_SPEC, 4),
    "best_val_precision":      round(BEST_PREC, 4),
    "best_val_f1":             round(BEST_F1, 4),
    "pixel_area_cm2_nominal":  PIXEL_AREA_CM2_NOMINAL,
    "area_note": (
        "ulcer_area_cm2 in XAI exports = predicted_pixels × pixel_area_cm2_nominal; "
        "calibrate for physical units."
    ),
    "augmentation":     "albumentations (elastic, shift-scale-rotate, grid-distort, CLAHE, GaussNoise, coarse-dropout)",
    "loss":             "Dice + BCEWithLogits (optional Focal); combined_loss = Dice+Focal",
    "xai_methods":      ["Score-CAM", "Attention Rollout", "RISE"],
}
cfg_path = EXPORT_DIR / f"{slug}_config.json"
with open(cfg_path, "w") as f:
    json.dump(model_config, f, indent=2)

print(f"Weights : {weights_path}  ({weights_path.stat().st_size / 1e6:.1f} MB)")
print(f"Config  : {cfg_path}")


In [ ]:
# ── 9.2  ONNX export ──────────────────────────────────────────────────────────
import torch.onnx
from copy import deepcopy

onnx_path    = EXPORT_DIR / f"{slug}.onnx"
export_model = deepcopy(BEST_MODEL).cpu().eval()
dummy        = torch.randn(1, 3, BEST_SIZE, BEST_SIZE)

try:
    torch.onnx.export(
        export_model, dummy, str(onnx_path),
        opset_version=17,
        input_names=["image"], output_names=["logits"],
        dynamic_axes={"image":  {0: "batch", 2: "H", 3: "W"},
                      "logits": {0: "batch", 2: "H", 3: "W"}},
    )
    print(f"ONNX export successful : {onnx_path}  ({onnx_path.stat().st_size/1e6:.1f} MB)")
except Exception as exc:
    print(f"ONNX export failed: {exc}")
    onnx_path = None
finally:
    del export_model


In [ ]:
# ── 9.3  Validate ONNX ────────────────────────────────────────────────────────
if onnx_path and onnx_path.exists():
    import onnx, onnxruntime as ort

    onnx.checker.check_model(onnx.load(str(onnx_path)))
    print("ONNX structure: valid ✓")

    session  = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    dummy_np = torch.randn(1, 3, BEST_SIZE, BEST_SIZE).numpy()
    outputs  = session.run(None, {session.get_inputs()[0].name: dummy_np})
    print(f"ONNX inference: output shape {outputs[0].shape}  ✓")

    BEST_MODEL.eval()
    with torch.no_grad():
        pt_out = BEST_MODEL(torch.tensor(dummy_np)).numpy()
    max_diff = float(np.abs(pt_out - outputs[0]).max())
    print(f"ONNX vs PyTorch max diff: {max_diff:.6f}  {'✓' if max_diff < 0.01 else '⚠ WARNING'}")
else:
    print("ONNX file not available — skipping validation.")


In [ ]:
# ── 9.4  Copy XAI + zip ───────────────────────────────────────────────────────
xai_export = EXPORT_DIR / "xai"
if xai_export.exists(): shutil.rmtree(xai_export)
shutil.copytree(str(XAI_DIR), str(xai_export))
print(f"XAI artefacts copied → {xai_export}  ({len(list(xai_export.iterdir()))} files)")

zip_target = OUTPUT_DIR / "dfu_seg_export"
zip_path   = OUTPUT_DIR / "dfu_seg_export.zip"
if zip_path.exists(): zip_path.unlink()

shutil.make_archive(str(zip_target), "zip", root_dir=str(EXPORT_DIR), base_dir=".")
print(f"✓ Export zip created : {zip_path}  ({zip_path.stat().st_size/1e6:.1f} MB)")

import zipfile
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
print(f"  {len(names)} files in archive.")


In [ ]:
# ── 9.5  Final summary ────────────────────────────────────────────────────────
print("=" * 72)
print("  GLUNOVA DFU — EXPORT SUMMARY  (v2 Optimised)")
print("=" * 72)
print()
print("  Dataset splits")
print(f"    Train : {len(train_imgs)}  |  Val : {len(val_imgs)}  |  Test : {len(test_imgs)}")
print()
print("  Model comparison (validation set — TTA + tuned threshold)")
print(f"  {'Model':<30}  {'Dice':>8}  {'IoU':>8}")
print(f"  {'-'*30}  {'--------':>8}  {'--------':>8}")
for name, md_, mi, *_ in MODEL_RESULTS:
    marker = "  ← selected" if name == BEST_NAME else ""
    print(f"  {name:<30}  {md_:8.4f}  {mi:8.4f}{marker}")
print()
print(f"  Prediction threshold : {BEST_THRESHOLD:.2f}  (tuned on val set)")
print()
print("  Optimisation")
print("    Augmentation : Albumentations (elastic, CLAHE, noise, dropout)")
print("    Loss         : Dice(50%) + BCE(50%); Focal available (combined_loss / dice_bce focal_w)")
print("    TTA          : h-flip + v-flip ensemble")
print("    LR schedule  : 5-epoch warmup → cosine anneal to 1e-7")
print("    Patience     : 10  |  Max epochs : 60")
print()
print("  XAI (segmentation-native)")
print(f"    Samples : {len(xai_records)}")
print( "    Methods : Score-CAM + Attention Rollout + RISE + PDF reports")
print()
print("  Export bundle")
print(f"    Zip    : {zip_path}")
print(f"    Size   : {zip_path.stat().st_size/1e6:.1f} MB")
print()
print("  Next step : Phase 2 — FastAPI + Docker inference server")
print("=" * 72)
